Please make sure you have read the [Getting Started page](https://pyrealm.readthedocs.io/en/latest/users/getting_started.html) before attempting to run this notebook.

# Isotopic discrimination

C3 and C4 plants assimilate the heavier atmospheric $\ce{^{13}CO2}$ and $\ce{^{14}CO2}$
molecules less easily than $\ce{^{12}CO2}$, leading to a discrimination against carbon
13 and carbon 14 and alteration of the resulting isotopic composition of plant tissues.
The isotopic discrimination and associated isotopic composition of a plant material
depends on the photosynthetic pathway.

The `pmodel` module provides the `CalcCarbonIsotopes` class, which takes the predicted
optimal chi ($\chi$) and photosynthetic pathway from a fitted `PModel` instance and
predicts various isotopic discrimination and composition values.

The predictions from the `CalcCarbonIsotopes` class are
driven by variation in $\chi$. The examples below show predictions across a range of
values of $\chi$. The sequence of $\chi$ values used is created by using the P Model to
estimate $\chi$ across a temperature gradient, giving the range of $\chi$ values shown
below for C3 and C4 plants.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pyrealm.pmodel import CalcCarbonIsotopes, PModelEnvironment
from pyrealm.pmodel.pmodel import PModel

# Use a simple temperature sequence to generate a range of optimal chi values
n_pts = 31
tc_1d = np.linspace(-10, 40, n_pts)

# Fit models. Since only relative light use efficiency is needed,
# fAPAR and PPFD are set to unity.
env = PModelEnvironment(tc=tc_1d, patm=101325, co2=400, vpd=1000, fapar=1, ppfd=1)
mod_c3 = PModel(env, method_optchi="prentice14")
mod_c4 = PModel(env, method_optchi="c4")

plt.plot(tc_1d, mod_c3.optchi.chi.flatten(), label="C3", marker=".")
plt.plot(tc_1d, mod_c4.optchi.chi.flatten(), label="C4", marker=".")
plt.title(r"Variation in $\chi$ with temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel(r"Predicted optimal $\chi$")
plt.legend(frameon=False)
plt.show()

## Calculation of values

The {class}`~pyrealm.pmodel.isotopes.CalcCarbonIsotopes` class takes a
{class}`~pyrealm.pmodel.pmodel.PModel` instance, along with estimates of the atmospheric
isotopic ratios for Carbon 13 ($\delta13C$, permil) and Carbon 14 ($\Delta14C$, permil)
and calculates the following predictions:

* `Delta13C_simple`: discrimination against carbon 13 ($\Delta\ce{^{13}C}$,
  permil) excluding photorespiration.
* `Delta13C`: discrimination against carbon 13 ($\Delta\ce{^{13}C}$, permil)
  including photorespiration.
* `Delta14C`: discrimination against carbon 14 ($\Delta\ce{^{14}C}$, permil)
  including photorespiration.
* `d13C_leaf`: isotopic ratio of carbon 13 in leaves ($\delta\ce{^{13}C}$,
  permil).
* `d14C_leaf`: isotopic ratio of carbon 14 in leaves ($\delta\ce{^{14}C}$,
  permil).
* `d13C_wood`: isotopic ratio of carbon 13 in wood ($\delta\ce{^{13}C}$,
  permil), given a parameterized post-photosynthetic fractionation.

The calculations differ between C3 and C4 plants, and this is set by the selection of
the `method_optchi` argument used for the {class}`~pyrealm.pmodel.pmodel.PModel`
instance.


In [ ]:
carb_c3 = CalcCarbonIsotopes(mod_c3, d13CO2=-8.4, D14CO2=19.2)
carb_c3.summarize()

In [ ]:
carb_c4 = CalcCarbonIsotopes(mod_c4, d13CO2=-8.4, D14CO2=19.2)
carb_c4.summarize()

The plots below show how the calculated values alter with $\chi$. The differences in the
direction of these relationships between C3 and C4 pathways creates a predictable
isotopic signature of relative contributions of the two pathways.

In [ ]:
# Create side by side subplots
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

attrs = [
    "Delta13C_simple",
    "Delta13C",
    "Delta14C",
    "d13C_leaf",
    "d14C_leaf",
    "d13C_wood",
]

for attr, ax in zip(attrs, axes.flatten()):
    ax.plot(mod_c3.optchi.chi, getattr(carb_c3, attr), label="C3")
    ax.plot(mod_c4.optchi.chi, getattr(carb_c4, attr), label="C4")
    ax.set_title(attr)
    ax.set_ylabel(attr)
    ax.set_xlabel("Optimal Chi (-)")
    ax.legend(frameon=False)

fig.tight_layout()